# Week 1 exercises: Images as numerical data

The aim is to practise **interpretation**, not Python syntax. Use NumPy documentation when needed.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, io

try:
    import oip
    oip_available = True
except Exception as exc:
    oip_available = False
    print('Could not import oip:')
    print(repr(exc))

print('oip available:', oip_available)


In [ ]:
gray = oip.load_gray()
rgb = oip.load_rgb()
print(gray.shape, gray.dtype)
print(rgb.shape, rgb.dtype)

## Problem 1.1 - Read an image array

Store the number of rows, columns, and RGB channels in `n_rows`, `n_cols`, and `n_channels`.

In [ ]:

n_rows, n_cols, n_channels = rgb.shape


print(n_rows, n_cols, n_channels)

## Problem 1.2 - Coordinates and a region of interest

Extract the central half of the grayscale image. The result should have approximately half the original height and half the original width. Store it in `roi`.

In [ ]:
h, w = gray.shape


r0, r1 = h // 4, 3 * h // 4
c0, c1 = w // 4, 3 * w // 4
roi = gray[r0:r1, c0:c1]


print('original:', gray.shape, 'ROI:', roi.shape)
plt.imshow(roi, cmap='gray')
plt.axis('off');

## Problem 1.3 - Spatial sampling

Create `sampled` by keeping every fourth row and every fourth column of `gray`.

Then answer in one sentence: what property of the digital image did you change - spatial sampling or quantisation?

In [ ]:

sampled = gray[::4, ::4]
answer_sampling = 'Spatial sampling: we kept fewer spatial positions while retaining the original stored values.'


print(sampled.shape)
print(answer_sampling)

## Problem 1.4 - Quantisation

The input is an 8-bit grayscale image. Quantise it to **4 intensity levels** while keeping values in the range 0 to 255. Store the result in `q4` as floating-point values.

In [ ]:
g = gray.astype(np.float64)


levels = 4
q4 = np.round(g / 255 * (levels - 1)) / (levels - 1) * 255


print('unique values:', np.unique(q4))
plt.imshow(q4, cmap='gray', vmin=0, vmax=255)
plt.axis('off');

## Problem 1.5 - A numerical comparison

Compute the root-mean-square (RMS) difference between the original grayscale image and `q4`. Store it in `rms_error`.

In [ ]:

rms_error = np.sqrt(np.mean((q4 - g)**2))


print('RMS error:', rms_error)

## Problem 1.6 - Element-wise or matrix multiplication?

Run the self-check below and choose the operation that matches each situation. Click **Check answers** when you are finished.


In [ ]:
# Problem 1.6 self-check
QUESTIONS = [['Which operation should be used to apply a binary mask to an image?', ['Matrix product', 'Element-wise multiplication', 'Dot product', 'Matrix inverse'], 1], ['Which operation represents the linear map $y=Ax$?', ['Element-wise multiplication', 'Matrix product', 'Pixel-wise clipping', 'Broadcast addition'], 1], ['You want to multiply each RGB channel by a separate gain factor. Which operation is conceptually appropriate?', ['Element-wise multiplication with broadcasting', 'Matrix inversion', 'A determinant', 'A matrix product between the image and itself'], 0], ['For two equally shaped image arrays `A` and `B`, what does `A * B` do?', ['Multiplies corresponding array elements', 'Computes a matrix product', 'Concatenates the arrays', 'Computes a dot product over all pixels'], 0]]


In [ ]:
import ipywidgets as widgets
from IPython.display import display

radio_widgets = []
output = widgets.Output()

quiz_boxes = []
for i, (question, options, correct) in enumerate(QUESTIONS):
    label = widgets.HTML(f"<b>Q{i+1}.</b> {question}")
    radio = widgets.RadioButtons(options=options, description='', disabled=False,
                                   layout=widgets.Layout(width='max-content'))
    radio_widgets.append(radio)
    quiz_boxes.append(widgets.VBox([label, radio]))

check_button = widgets.Button(description='Check answers', button_style='success')

def check_answers(b):
    output.clear_output()
    with output:
        score = 0
        for i, (question, options, correct) in enumerate(QUESTIONS):
            chosen = radio_widgets[i].value
            chosen_index = options.index(chosen) if chosen in options else -1
            if chosen_index == correct:
                score += 1
                print(f"Q{i+1}: Correct!")
            else:
                print(f"Q{i+1}: Not quite. Correct answer: {options[correct]}")
        print(f"\nScore: {score} / {len(QUESTIONS)}")

check_button.on_click(check_answers)

display(widgets.VBox(quiz_boxes + [check_button, output]))

## Problem 1.7 - Data type and arithmetic

Explain why converting an unsigned 8-bit image to floating point before a sequence of arithmetic operations is often useful. Keep the answer to two or three sentences.

In [ ]:

answer_dtype = (
    'Unsigned 8-bit values have a limited range, so arithmetic can overflow, underflow, '
    'or clip in ways that obscure the intended mathematics. Converting to floating point '
    'makes intermediate values explicit; we can clip and convert back only when required.'
)


print(answer_dtype)

## Problem 1.8 - Optional challenge: Matrix-vector mapping

Consider

$$
A=\begin{bmatrix}1&1\\1&-1\end{bmatrix},\qquad x=\begin{bmatrix}3\\1\end{bmatrix}.
$$

Compute $y=Ax$ using NumPy and interpret the shape of the result.

In [ ]:
A = np.array([[1.0, 1.0], [1.0, -1.0]])
x = np.array([3.0, 1.0])


y = A @ x


print('y =', y, 'shape =', y.shape)

## Reflection

- image data have shape, type, and range;
- slicing changes which spatial samples we keep;
- quantisation changes which intensity values are available;
- array arithmetic expresses image-processing operations;
- norms turn differences into numerical measures.